In [6]:
# prompt: import pandas, numpy, streamlit, joblib, and other libraries required for machine learning model such as randomforest classifier and streamlit application embedding

!pip install streamlit scikit-learn

  Using cached streamlit-1.47.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached watchdog-6.0.0-py3-none-manylinux2014_x86_64.whl.metadata (44 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
Using cached streamlit-1.47.0-py3-none-any.whl (9.9 MB)
Using cached pydeck-0.9.1-py2.py3-none-any.whl (6.9 MB)
Using cached watchdog-6.0.0-py3-none-manylinux2014_x86_64.whl (79 kB)


In [7]:
import pandas as pd
import numpy as np
import streamlit as st
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

In [12]:
from google.colab import files

uploaded = files.upload()

if uploaded:
  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
else:
  print("File upload cancelled or failed.")

Saving adult 3.csv to adult 3.csv
User uploaded file "adult 3.csv" with length 5326370 bytes


In [13]:
# prompt: load adult 3.csv file from contents

data = pd.read_csv('adult 3.csv')
print(data.head())

   age  workclass  fnlwgt     education  educational-num      marital-status  \
0   25    Private  226802          11th                7       Never-married   
1   38    Private   89814       HS-grad                9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm               12  Married-civ-spouse   
3   44    Private  160323  Some-college               10  Married-civ-spouse   
4   18          ?  103497  Some-college               10       Never-married   

          occupation relationship   race  gender  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                  ?    Own-child  White  Female             0             0   

   hours-per-week native-country incom

In [14]:
data.shape

(48842, 15)

In [15]:
data.isna().sum()

,0
age,0
workclass,0
fnlwgt,0
education,0
educational-num,0
marital-status,0
occupation,0
relationship,0
race,0
gender,0


In [16]:
# Replace '?' with 'Not-listed' in the specified columns
data['workclass'] = data['workclass'].replace('?', 'Not-listed')
data['occupation'] = data['occupation'].replace('?', 'Not-listed')
data['native-country'] = data['native-country'].replace('?', 'Not-listed')

In [17]:
data.drop(columns=['education'], inplace=True)

In [18]:
# Apply LabelEncoder to categorical features, excluding the target variable 'income'

# Identify categorical columns (excluding 'income' and already numerical columns)
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']

le = LabelEncoder()

for col in categorical_cols:
    # Check if the column exists in the DataFrame before encoding
    if col in data.columns:
        data[col] = le.fit_transform(data[col])
    else:
        print(f"Warning: Categorical column '{col}' not found in the DataFrame.")

# Note: The 'income' column will be handled separately for model training
# and should be encoded if it contains categorical values (like '>50K', '<=50K').
# The original cell applied encoding to all columns, including numerical.
# This modified cell applies it only to identified categorical columns.

# Display the first few rows of the modified data to see the changes
display(data.head())

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,4,226802,7,4,6,3,2,1,0,0,40,39,<=50K
1,38,4,89814,9,2,4,0,4,1,0,0,50,39,<=50K
2,28,1,336951,12,2,11,0,4,1,0,0,40,39,>50K
3,44,4,160323,10,2,6,0,2,1,7688,0,40,39,>50K
4,18,3,103497,10,4,7,3,4,0,0,0,30,39,<=50K


In [19]:
xtrain, xtest, ytrain, ytest = train_test_split(data.drop(columns=['income']), data['income'], test_size=0.2, random_state=42)
xtrain

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country
37193,42,4,145175,10,2,6,0,4,1,0,0,40,39
31093,52,6,175029,6,2,2,0,4,1,0,0,35,39
33814,34,1,172664,9,2,2,0,4,1,0,0,40,39
14500,28,4,125791,9,4,0,1,4,0,0,0,40,39
23399,46,4,28419,11,4,14,1,4,1,0,0,50,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,36,4,635913,9,3,8,1,2,1,0,0,40,39
44732,34,4,107624,10,2,2,0,4,1,0,0,50,39
38158,28,4,250135,10,0,3,1,4,0,0,0,40,39
860,46,7,96652,11,5,0,4,2,0,0,0,40,39


In [20]:
models = {
    'LogisticRegression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Support Vector Machine': SVC()
}

results = {}

for name, model in models.items():
  pipe = Pipeline([
      ('scaler', StandardScaler()),
      ('model', model)
  ])
  pipe.fit(xtrain, ytrain)
  ypred = pipe.predict(xtest)
  acc = accuracy_score(ytest, ypred)
  results[name] = acc

for name, acc in results.items():
  print(f"{name}: {acc}")

LogisticRegression: 0.8280274337189067
Random Forest: 0.8647763332992118
Gradient Boosting: 0.8725560446309756
K-Nearest Neighbors: 0.8346811342000204
Support Vector Machine: 0.8564847988535162


In [21]:
joblib.dump(models['Random Forest'], 'random_forest_model.pkl')



['random_forest_model.pkl']

In [61]:
# Display the classes learned by the label encoder
print("Label Encoder Classes:")
for col in ['workclass', 'occupation', 'native-country', 'marital-status', 'relationship', 'race', 'gender']:
    if col in data.columns:
        # To access the classes for each column, we need to refit or store individual encoders
        # Since we fitted the encoder within the loop in cell 9yH_onVDz_rb,
        # the 'le' variable currently holds the encoder fitted on the last column ('gender').
        # To see classes for all columns, we would need to fit and store encoders for each.
        # However, the ValueError specifically mentions 'Private' for 'workclass'.
        # Let's refit a temporary encoder specifically for 'workclass' to inspect its classes.
        temp_le = LabelEncoder()
        temp_le.fit(data[col].unique())
        print(f"- {col}: {temp_le.classes_}")

# Note: The 'le' variable saved in the pickle file is the one fitted on the last column ('gender')
# in the loop in cell 9yH_onVDz_rb. A better approach would be to use a dictionary
# to store encoders for each column and save that dictionary.
# Let's modify cell 9yH_onVDz_rb to save a dictionary of encoders.

Label Encoder Classes:
- workclass: [0 1 2 3 4 5 6 7 8]
- occupation: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
- native-country: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41]
- marital-status: [0 1 2 3 4 5 6]
- relationship: [0 1 2 3 4 5]
- race: [0 1 2 3 4]
- gender: [0 1]


In [76]:
%%writefile app.py
import pandas as pd
import numpy as np
import streamlit as st
import joblib
from sklearn.preprocessing import StandardScaler

# Load the trained model
model = joblib.load('random_forest_model.pkl')

st.set_page_config(page_title='Employee Salary Prediction', page_icon=':money_with_wings:', layout='wide')
st.title('Employee Salary Prediction')
st.markdown('Lets try to predict an employee\'s salary')

# Based on the output of cell e7e6b718, the mappings are as follows:
st.sidebar.header('Employee Details')

workclass_map = {'Not-listed': 0, 'Federal-gov': 1, 'Local-gov': 2, 'Never-worked': 3, 'Private': 4, 'Self-emp-inc': 5, 'Self-emp-not-inc': 6, 'State-gov': 7, 'Without-pay': 8}
marital_status_map = {'Married-AF-spouse': 0, 'Married-civ-spouse': 1, 'Married-spouse-absent': 2, 'Never-married': 3, 'Separated': 4, 'Divorced': 5, 'Widowed': 6}
occupation_map = {'Adm-clerical': 0, 'Armed-Forces': 1, 'Craftsman-repair': 2, 'Exec-managerial': 3, 'Farming-fishing': 4, 'Handlers-cleaners': 5, 'Machine-op-inspct': 6, 'Other-service': 7, 'Priv-house-serv': 8, 'Prof-specialty': 9, 'Protective-serv': 10, 'Sales': 11, 'Tech-support': 12, 'Transport-moving': 13, 'Not-listed': 14}
relationship_map = {'Husband': 0, 'Not-in-family': 1, 'Other-relative': 2, 'Own-child': 3, 'Unmarried': 4, 'Wife': 5}
race_map = {'Amer-Indian-Eskimo': 0, 'Asian-Pac-Islander': 1, 'Black': 2, 'Other': 3, 'White': 4}
gender_map = {'Female': 0, 'Male': 1}
native_country_map = {'Cambodia': 0, 'Canada': 1, 'China': 2, 'Columbia': 3, 'Cuba': 4, 'Dominican-Republic': 5, 'Ecuador': 6, 'El-Salvador': 7, 'England': 8, 'France': 9, 'Germany': 10, 'Greece': 11, 'Guatemala': 12, 'Haiti': 13, 'Honduras': 14, 'Hong': 15, 'Hungary': 16, 'India': 17, 'Iran': 18, 'Ireland': 19, 'Italy': 20, 'Jamaica': 21, 'Japan': 22, 'Laos': 23, 'Mexico': 24, 'Nicaragua': 25, 'Outlying-US(Guam-USVI-etc)': 26, 'Peru': 27, 'Philippines': 28, 'Poland': 29, 'Portugal': 30, 'Puerto-Rico': 31, 'Scotland': 32, 'South': 33, 'Taiwan': 34, 'Thailand': 35, 'Trinadad&Tobago': 36, 'United-States': 37, 'Vietnam': 38, 'Yugoslavia': 39, 'Not-listed': 40}


# Create input fields for each feature using selectbox for categorical and number_input for numerical
age = st.sidebar.slider('Age', min_value=17, max_value=90, value=30)
workclass = st.sidebar.selectbox('Workclass', list(workclass_map.keys())) # Use keys for display
fnlwgt = st.sidebar.slider('Fnlwgt', min_value=10000, max_value=1500000, value=200000)
educational_num = st.sidebar.number_input('Educational-num', min_value=1, max_value=16, value=9) # Using educational-num directly
marital_status = st.sidebar.selectbox('Marital-status', list(marital_status_map.keys())) # Use keys for display
occupation = st.sidebar.selectbox('Occupation', list(occupation_map.keys())) # Use keys for display
relationship = st.sidebar.selectbox('Relationship', list(relationship_map.keys())) # Use keys for display
race = st.sidebar.selectbox('Race', list(race_map.keys())) # Use keys for display
gender = st.sidebar.selectbox('Gender', list(gender_map.keys())) # Use keys for display
capital_gain = st.sidebar.slider('Capital Gain', min_value=0, max_value=100000, value=0)
capital_loss = st.sidebar.slider('Capital Loss', min_value=0, max_value=100000, value=0)
hours_per_week = st.sidebar.number_input('Hours per Week', min_value=1, max_value=100, value=40)
native_country = st.sidebar.selectbox('Native Country', list(native_country_map.keys())) # Use keys for display

# Create a DataFrame from user inputs
input_data = pd.DataFrame({
    'age':[age],
    'workclass':[workclass_map[workclass]], # Map back to numerical value
    'fnlwgt':[fnlwgt],
    'educational_num':[educational_num],
    'marital_status':[marital_status_map[marital_status]], # Map back to numerical value
    'occupation':[occupation_map[occupation]], # Map back to numerical value
    'relationship':[relationship_map[relationship]], # Map back to numerical value
    'race':[race_map[race]], # Map back to numerical value
    'gender':[gender_map[gender]], # Map back to numerical value
    'capital_gain':[capital_gain],
    'capital_loss':[capital_loss],
    'hours_per_week':[hours_per_week],
    'native_country':[native_country_map[native_country]] # Map back to numerical value
    })

st.write("### Input Data")
st.write(input_data)

if st.button('Predict Salary'):

    # Make prediction using the processed data
    prediction = model.predict(input_data)

    # Display prediction
    if prediction[0] == '>50K':
        st.success('Predicted Salary: >50K')
    else:
        st.success('Predicted Salary: <=50K')

Overwriting app.py


In [63]:
# Install pyngrok
!pip install pyngrok

In [71]:
!ngrok authtoken 30GLeSyeozuwxagVo3qd8hsLW7k_44EfAR3H1wkNkVwEhc8as

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [77]:
import os
import threading

def run_streamlit():
  os.system('streamlit run app.py --server.port 8501')

thread = threading.Thread(target=run_streamlit)
thread.start()

In [78]:
from pyngrok import ngrok
import time

# Terminate any existing ngrok tunnels
ngrok.kill()

time.sleep(5)

public_url = ngrok.connect(8501)
print("Your streamlit app is live here:", public_url)

Your streamlit app is live here: NgrokTunnel: "https://9a152c7a5712.ngrok-free.app" -> "http://localhost:8501"
